# Creating an Official Test Set

Our test set must be made of entirely "un-processed" data. ie, no imputation to get missing properties. It needs to be as large as possible, because data can be removed from the test set, but not added. It also needs to be robust against any new data we do add. 

In this script, we will use the following procedure:

    1) Determine how many points contain our target. This is our entire dataset for that target. The test size is 20% of this length. 
    2) Isolate points that contain our target and all inputs.
    3) Take all test data from isolated points from 2)
    4) Recombine remaining full datapoints with those missing some data to use as training set
    
This involves 3 inter-related dataframes: 

    1) df_y, the dataframe containing only points with tagret data. This determines test size
    2) df_x_y the dataframe containing only points with complete data (filters+other_xs+target). We choose test data from this set
    3) df, initialy contains all data and is processed throughout this script
So, first we load our data:

In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.append('..')
from Utilities import custom
from sklearn.model_selection import train_test_split
from IPython.core.magic import register_cell_magic
import matplotlib as mpl
import matplotlib.pyplot as plt

@register_cell_magic
def skip(line, cell):
    return

combined_data='../Datasets/Original_Datasets/NGC_combi.csv'
df_full=pd.read_csv(combined_data)
df_full=df_full.set_index('Star ID') #this is a unique identifier for each star, instead of each cluster


### Remove Repeat Data

Taking all repeate data completely out of both test and train set.

In [2]:
isduped=df_full.index.duplicated(keep=False)
duped_ids=pd.DataFrame(df_full.iloc[isduped].index.unique())
#saving IDs to file
duped_ids.to_csv('../Datasets/duped_ids.csv', index=False)

#removing repeats from dataset
df=df_full.iloc[~isduped]

### NEED TO CHECK THAT NONE OF THE ABOVE ARE NEEDED

Any -99.9999 or +99.9999 type values have been properly replaced with NaNs already. Need to fix a few column names:

In [3]:
y_values=['O/Fe', 'Na/Fe']
filters=['F275W_abs', 'F336W_abs', 'F438W_abs', 'F606W_abs', 'F814W_abs']
other_xs=['age_Kruijssen', 'Fe/H']
df.rename(columns={'F435W': 'F438W', 'F435W_abs':'F438W_abs', 'F435W_e':'F438W_e'}, inplace=True)

/tmp/ipykernel_27554/1212626405.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'F435W': 'F438W', 'F435W_abs':'F438W_abs', 'F435W_e':'F438W_e'}, inplace=True)


### Split Dataset

Some of our clusters are small enough that we cannot evenly split them at the chosen percentage. I made a function that handles this for us automatically. It actually ended up being less important than I thought, because it (so far) only catches clusters with 1 member, since our test data needs to be such a larger percentage of our complete datapoints. Will probably matter more (I hope) when selecting train/test for single targets. 

In [4]:
df_y=df.dropna(subset=y_values, how='any')
test_size=np.int32(np.ceil(0.2*df_y.shape[0]))
print(f'dataset size: {df_y.shape}, test size: {test_size}')

dataset size: (527, 41), test size: 106


In [7]:
df_x_y=df.dropna(subset=y_values+filters+other_xs)
train_complete_points, test=custom.train_test_split(df_x_y, test_size=test_size) #this gives us the portion of training data with complete points ONLY
train_complete_points.describe()

,NGC,RAJ2000,DEJ2000,Distance (pc),neg_sigma_dist,pos_sigma_dist,Vmag,Vmag_abs,Bmag,Bmag_abs,...,F336W_e,F438W,F438W_abs,F438W_e,F606W,F606W_abs,F606W_e,F814W,F814W_abs,F814W_e
count,189.000000,189.000000,189.000000,189.000000,189.000000,189.000000,153.000000,153.000000,89.000000,91.000000,...,182.000000,189.000000,189.000000,169.000000,189.000000,189.000000,155.000000,189.000000,189.000000,153.000000
mean,4813.777778,202.735693,-35.498570,8489.312169,-93.407407,94.174603,14.147170,-0.080492,15.739787,0.989069,...,0.007724,15.633904,1.186108,0.004376,13.927328,-0.520468,0.000112,12.877383,-1.570413,0.000126
std,2042.800397,84.337816,26.209948,3424.776437,47.081167,47.555636,1.285089,1.323469,1.205767,2.750379,...,0.013487,1.296943,1.227915,0.012793,1.297390,1.330088,0.001326,1.345754,1.403883,0.001560
min,104.000000,6.028079,-72.073897,1851.000000,-190.000000,15.000000,11.415000,-2.616338,12.967000,-15.240460,...,0.000000,13.040400,-1.161438,0.000000,11.034800,-2.883138,0.000000,9.838700,-4.195690,0.000000
25%,3201.000000,154.383704,-47.510955,5426.000000,-115.000000,47.000000,13.159000,-1.099393,15.149000,0.261607,...,0.000000,14.768900,0.346798,0.000000,13.044700,-1.433093,0.000000,11.957200,-2.582593,0.000000
50%,5139.000000,201.746708,-44.751499,8988.000000,-95.000000,96.000000,14.285000,-0.145002,15.642000,1.200998,...,0.004050,15.552800,0.919710,0.000000,13.984200,-0.641390,0.000000,12.936200,-1.754799,0.000000
75%,6388.000000,264.057896,-23.164400,10404.000000,-47.000000,116.000000,14.990000,0.815540,16.598000,2.300574,...,0.010500,16.307900,2.149940,0.004900,14.618700,0.399955,0.000000,13.579800,-0.643510,0.000000
max,7099.000000,325.089600,36.475033,16400.000000,-16.000000,192.000000,17.249000,3.652968,17.909000,4.722968,...,0.101700,21.028600,6.260285,0.142600,20.206900,5.438585,0.016500,19.610800,4.842485,0.019300


In [8]:
set(test['NGC']).issubset(set(train_complete_points['NGC']))

True

In [9]:
test['NGC'].isin(train_complete_points['NGC']).all()

np.True_

This is False because any clusters with only 1 member are added to the testing data only

In [10]:
set(train_complete_points['NGC']).issubset(set(test['NGC']))

True

In [11]:
train_complete_points['NGC'].isin(test['NGC']).all()

np.True_

### Save Training and Testing Data

Now that we have our full test set identified, we just need to remove these instances from our training data by ID. We don't just save the "train_complete_points" variable because it only has complete points in it; we want to include incomplete points in training

In [12]:
common_indeces = df_y.index.intersection(test.index)
train=df_y.drop(common_indeces) #This gives us the ENTIRE training set available for this target, not just the training data with complete points
train

,NGC,Paper,Author Reference,RAJ2000,DEJ2000,Distance (pc),neg_sigma_dist,pos_sigma_dist,Vmag,Vmag_abs,...,F336W_e,F438W,F438W_abs,F438W_e,F606W,F606W_abs,F606W_e,F814W,F814W_abs,F814W_e
Star ID,,,,,,,,,,,,,,,,,,,,,
R0018794,104,Carretta,26058,6.028079,-72.073897,4521,-31,31,12.403,-0.873173,...,0.0000,13.7323,0.456127,0.0000,12.0009,-1.275273,0.0,10.9820,-2.294173,0.0
R0025327,104,Carretta,30340,6.011504,-72.055008,4521,-31,31,12.395,-0.881173,...,0.0000,13.8185,0.542327,0.0000,12.0301,-1.246073,0.0,10.9334,-2.342773,0.0
R0001617,288,Carretta,200001,13.171308,-26.557431,8988,-88,89,11.380,-3.388315,...,0.0000,NaN,NaN,NaN,12.0358,-2.732515,0.0,10.6808,-4.087515,0.0
R0001265,288,Carretta,200014,13.208867,-26.570667,8988,-88,89,13.864,-0.904315,...,0.0006,15.0652,0.296885,0.0072,13.5954,-1.172915,0.0,12.6846,-2.083715,0.0
R0000667,288,Carretta,200016,13.177167,-26.586783,8988,-88,89,13.883,-0.885315,...,0.0073,15.0788,0.310485,0.0029,13.6055,-1.162815,0.0,12.7164,-2.051915,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
R0000271,7099,Carretta,8757,325.081788,-23.198703,8458,-89,90,17.249,2.612662,...,0.0097,17.9063,3.269962,0.0058,17.0129,2.376562,0.0,16.3584,1.722062,0.0
R0004711,7099,Carretta,9817,325.099733,-23.160014,8458,-89,90,14.720,0.083662,...,0.0036,15.6192,0.982862,0.0007,14.5634,-0.072938,0.0,13.7918,-0.844538,0.0
R0002313,7099,Carretta,10831,325.097229,-23.180417,8458,-89,90,13.437,-1.199338,...,0.0000,14.4706,-0.165738,0.0000,13.1491,-1.487238,0.0,12.2758,-2.360538,0.0


And now we save our training data and testing data seperately:

In [13]:
train.to_csv('../Datasets/training_data.csv')
test.to_csv('../Datasets/TEST_DATA.csv')

And we also combine the indeces with the repeated indeces to create a blacklist:

In [14]:
blacklist=list(duped_ids)
blacklist.extend(list(common_indeces))

blacklist=pd.DataFrame({'Star ID':blacklist})
blacklist.to_csv('../Datasets/TESTING_OR_DUPLICATE_IDS.csv', index=False)

# Different Train/Test Sets for Each Target

Very limited data. Should try making test/train sets for each desired output. Basically doing first part of script, but in loop:

In [15]:
for target in y_values:
    blacklist=list(duped_ids)
    df_y=df.dropna(subset=target)
    test_size=np.int32(np.ceil(0.2*df_y.shape[0]))
    #drop all rows missing inputs or abundance
    df_x_y=df.dropna(subset=filters+other_xs+[target])
    #separate complete points into test and train
    train_complete_points, test=custom.train_test_split(df_x_y, test_size, target=target)
    print(train_complete_points['NGC'].isin(test['NGC']).all(), test['NGC'].isin(train_complete_points['NGC']).all())
    #create test data with full coverage for this abundance
    common_indeces = df_y.index.intersection(test.index)
    #Drop these test points from the full dataset
    train=df_y.drop(common_indeces)
    print(target, df_y.shape, train.shape, test.shape, test.shape[0]/df_y.shape[0])
    #train.to_csv(f'../Datasets/{target.replace('/', '_')}_training_data.csv')
    #test.to_csv(f'../Datasets/{target.replace('/', '_')}_TEST_DATA.csv')
    blacklist.extend(list(common_indeces))
    blacklist=pd.DataFrame({'Star ID':blacklist})
    #blacklist.to_csv(f'../Datasets/{target.replace('/', '_')}_TESTING_OR_DUPLICATE_IDs.csv', index=False)

True True
O/Fe (566, 41) (451, 41) (115, 41) 0.20318021201413428
True True
Na/Fe (683, 41) (545, 41) (138, 41) 0.2020497803806735
